In [1]:
import pandas as pd
import entsoe
import cdsapi
from dotenv import load_dotenv
import os
import time
import calendar


_ = load_dotenv()

# Data Acquisition
In our first step we need to gather the data we will be working with.
Please execute the cell above this. 
It will import the needed packages and load your API keys into your environment.

----

We will be using two major data sources: the ENTSO-E Transparency Platform [1] for all data regarding the enrgy markets and the ERA5/Copernicus Dataset [2] for weather data. These datasets are quite extensive, accurate and easily acquired.
We will download the data and store them in a file to be processed and analysed in later steps.

## ENTSO-E

We are starting with the ENTSO-E Datasets as they are the ones we are primarily trying to analyze.
Since I am based in germany we will only be using the data of Germany and since I want to consider the effect of renewable energy I will also include Denmark.
However the principals laid out in this project should be adaptable to most other european countries.
We will only be using data from the years 2019 until 2025. This includes major market disruptions due to the ukraine war and the COVID-19 Pandemic.
The Datasets we will be using are:
  - Day-Ahead Prices
  - Actual Total Load
  - Aggregated Generation per Type

The reason to choose these is that energy markets are using the _Merit-Order-Model_ [3] to choose the energy prices.
This model orders the different generators from cheapest running cost to highest running cost.
That is why the Aggregated Generation per Type is interesting to us.
The model then checks what the cheapest set of generators are which will still cover the demand.
That is why the Actual Total Load is interesting.
Finally the price of energy is determined by the running cost of the most expensive generator needed to cover demand.
All other generators are able to sell their 'cheaper' energy at the more expensive price.

In [ ]:
OUTPUT_DIR = 'data/raw/entsoe'
YEARS = range(2019,2025+1)
COUNTRIES = ['DE_LU', 'DK1', 'DK2']


os.makedirs(OUTPUT_DIR, exist_ok=True)
client = entsoe.EntsoePandasClient(api_key=os.environ['ENTSOE_API_KEY'])

for country in COUNTRIES:
    for year in YEARS:
        start = pd.Timestamp(f'{year}0101', tz='UTC')
        end = pd.Timestamp(f'{year}1231', tz='UTC')
        country_code = 'DE_LU'     #  The bidding zone for Germany is the same as for Luxembourg
        target = os.path.join(OUTPUT_DIR, f'{country}_Price_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            print(f'requesting: {country}, {year}, price -> {target}')
            df = client.query_day_ahead_prices(country_code, start, end)
            print('received')
            df.to_frame(name="price").to_parquet(target)
        target = os.path.join(OUTPUT_DIR, f'{country}_Generation_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            print(f'requesting: {country}, {year}, generation -> {target}')
            df = client.query_generation(country_code=country_code, start=start, end=end, nett=False)
            print('received')
            df.to_parquet(target)
        target = os.path.join(OUTPUT_DIR, f'{country}_Load_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            print(f'requesting: {country}, {year}, load -> {target}')
            df = client.query_load(country_code=country_code, start=start, end=end)
            print('received')
            df.to_frame(name="load").to_parquet(target)

requesting data/raw/entsoe/DE_LU_Price_2019.parquet


## Copernicus
Weather data is hugely important for the energy markets,
it directly influences both sides of the Merit-Order-Model: the generation capacity of wind, solar and water energy directly correspond to the weather you are having (or had).
But also the energy consumption changes dramatically with the weather. If it is cold people will be consuming more energy to heat.
While on particularly hot days people might be more prone to turning on air conditioning.

In particularly extreme cases weather can even produce outages and disrupt the entire energy network.

_Note: I am unsure of how much industrial energy consumption varies with the weather._

This step is expected to take a long while since the amount of data is very large.
I suggest you to proceed to the next notebooks and only go through the steps which use only the ENTSO-E data and leave this runnning in the background in the meanwhile.

In [ ]:
DATASET = 'reanalysis-era5-single-levels'

OUTPUT_DIR = 'data/raw/era5'

# rough bounding boxes
COUNTRY_AREAS = { # [North, West, South, East]
    'germany': [55.1, 5.8, 47.2, 15.1],
    'luxembourg': [50.2, 5.7, 49.4, 6.5],
    'denmark': [57.8, 8.0, 54.5, 15.2],
}

VARIABLES = [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '100m_u_component_of_wind',
    '100m_v_component_of_wind',
    '2m_temperature',
    'surface_solar_radiation_downwards',
]

YEARS = range(2019,2025+1)
MONTHS = range(1, 13)
ALL_HOURS = [f'{h:02d}:00' for h in range(24)]


def build_request(area: list[float], year: int, month: int) -> dict:
    days_per_month = {
        month: [f'{d:02d}' for d in range(1, calendar.monthrange(year, month)[1] + 1)]
        for month in range(1, 13)
    }
    all_days = [f'{d:02d}' for d in range(1, 32)]

    return {
        'product_type': ['reanalysis'],
        'variable': VARIABLES,
        'year': [str(year)],
        'month': [f'{month:02d}'],
        'day': all_days,
        'time': ALL_HOURS,
        'area': area,  # [North, West, South, East]
        'data_format': 'netcdf',
        'download_format': 'unarchived',
    }


def download_all() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    client = cdsapi.Client(url=os.environ['CDS_API_URL'], key= os.environ['CDS_API_KEY'])

    for country, area in COUNTRY_AREAS.items():
        for year in YEARS:
            for month in MONTHS:
                target = os.path.join(OUTPUT_DIR, f'era5_{country}_{year}_{month}.nc')
                if os.path.exists(target):
                    print(f'skipping {target} (already exists)')
                    continue
    
                request = build_request(area, year, month)
                print(f'requesting: {country}, {year}, {month} -> {target}')
                client.retrieve(DATASET, request, target)


download_all()


requesting: germany, 2019, 1 -> data/raw/era5/era5_germany_2019_1.nc


2026-09-08 19:46:40,968 INFO Request ID is 27ed55f2-f982-4c2a-9a6e-27635e202ff1
2026-09-08 19:46:41,038 INFO status has been updated to accepted
2026-09-08 19:46:54,893 INFO status has been updated to running
2026-09-08 19:47:02,625 INFO status has been updated to successful


requesting: germany, 2019, 2 -> data/raw/era5/era5_germany_2019_2.nc


2026-09-08 19:47:05,797 INFO Request ID is c482cbaa-7dc8-4979-a149-cc490f9de4b2
2026-09-08 19:47:05,919 INFO status has been updated to accepted
2026-09-08 19:47:39,697 INFO status has been updated to successful


requesting: germany, 2019, 3 -> data/raw/era5/era5_germany_2019_3.nc


...

requesting: germany, 2019, 6 -> data/raw/era5/era5_germany_2019_6.nc


2026-09-08 19:49:13,963 INFO Request ID is 56212224-3541-486b-b3b1-32a6d0cdd4c3
2026-09-08 19:49:14,064 INFO status has been updated to accepted
2026-09-08 19:49:28,611 INFO status has been updated to running
2026-09-08 19:49:36,292 INFO status has been updated to successful


requesting: germany, 2019, 7 -> data/raw/era5/era5_germany_2019_7.nc


2026-09-08 19:49:38,385 INFO Request ID is 68495d0f-31fa-43f1-8699-125190a3b854
2026-09-08 19:49:38,465 INFO status has been updated to accepted


KeyboardInterrupt: 

## Sources
[1] ENTSO-E, "ENTSO-E Transparency Platform," 2026. [Online]. Available: https://transparency.entsoe.eu/. [Accessed: 07.09.2026].
  
[2] H. Hersbach et al., "ERA5 hourly data on single levels from 1940 to present",
    Copernicus Climate Change Service (C3S) Climate Data Store (CDS), 2018.
    [Online]. Available: https://doi.org/10.24381/cds.adbb2d47.
    [Accessed: 08.09.2026].
    
[3] F. Sensfuß, M. Ragwitz, and M. Genoese, "The merit-order effect: A detailed
    analysis of the price effect of renewable electricity generation on spot
    market prices in Germany," Energy Policy, vol. 36, no. 8, pp. 3076–3084, 2008.